# Phase-diagram Python API

This tutorial evaluates a voltage- and temperature-dependent surface phase diagram from the maintained SCAN+rVV10+U LiNiO2 (001) example data. It deliberately uses the public Python API in separate steps so that the thermodynamic assumptions, intermediate objects, and numerical results can be inspected.

In [ ]:
import json
from pathlib import Path

from IPython.display import JSON
from matplotlib import pyplot as plt

from surface_pd.configuration import PhaseDiagramConfiguration
from surface_pd.plot import plot_phase_diagram

The notebook expects to be started from the `examples/` directory.  The contents of the entire JSON configuration file can be explored in the tree below.

In [ ]:
configuration_path = Path("plotting-examples/lno-001-scan/charge.json")

with open(configuration_path) as fp:
    config = json.load(fp)

JSON(config)

The JSON configuration file contains details of the data to be analyzed, such as the calculation method used, as well as parameters for the phase diagram evaluation and visualization. 

## 1. Load the configuration

The `PhaseDiagramConfiguration` class can parse JSON config files for phase diagram evaluation.

The config file schema is version 1 to allow for extensions in the future. The calculation method used in this example data set is density-functional theory with the SCAN exchange--correlation functional, the revised Van-Voorhis 2010 dispersion correction, and an additional Hubbard-$U$ correction for the nickel $d$ band.

In [ ]:
configuration = PhaseDiagramConfiguration.read_json(configuration_path)

print("schema version:", configuration.schema_version)
print("method:", configuration.calculation_method)
print("component basis:", configuration.components)

We want to model how the LiNiO$_2$(001) surface changes depending on the charge voltage and the temperature.  The relevant chemical potentials are those of Li, which determines the cell voltage, and O, which is strongly temperature-dependent.  These two chemical potentials are therefore chosen to be *independent*.  The chemical potential of Ni can then be deduced from the Li and O chemical potentials and is, hence, *dependent*.

Specifically, for this example, the independent Li chemical potential is

$$\mu_{\mathrm{Li}}(V) = E_{\mathrm{Li(bcc)}} - e\,V$$

where the energy of Li in the ideal BCC structure, $E_{\mathrm{Li(bcc)}}$, is the reference energy and $e$ is the elementary charge (each Li$^+$ carries one charge), and $V$ is the cell potential. A preserved fixed-pressure approximation is used for the O chemical potential, $\mu_{\mathrm{O}}(T)$, where $T$ is the temperature. 

In the config file, this is encoded as (all energies are in electronvolts, eV)

In [ ]:
print(
    "independent components:",
    json.dumps(config["independent_chemical_potentials"], indent=4),
)

The dependent nickel chemical potential is fixed by the bulk reference equality

$$\mu_{\mathrm{Li}} + \mu_{\mathrm{Ni}} + 2\mu_{\mathrm{O}} = g_{\mathrm{LiNiO_2}}.$$

The model solves this equality rather than assigning a special meaning to nickel.

`PhaseDiagramConfiguration` automatically recognizes that the Ni chemical potential depends on the others.

In [ ]:
print("dependent components:", configuration.model.dependent_components)

## 2. Alignment of multiple DFT energy data sets

For each polar LiNiO2 surface, the maintained examples combine energies from two data sets: Li-terminated and Ni-terminated surface slab models.  If both are well-converged with regards to the slab thickness, the energies are compatible.  Both data sets are set up in the config file under `datasets`:

In [ ]:
print(json.dumps(config['datasets'], indent=2))

For each of the data sets, a path to the data files with DFT energies is given.  The table uses canonical columns for the phase ID, each declared component, the total DFT energy, and the surface area. Optional `column_overrides` are needed only for external tables with different headings. The dataset-level `number_of_surfaces` determines the normalization. For symmetric slab models with two equivalent surfaces, the grand potential is divided by twice the surface area.

In practice, numerical differences can occur for moderate numbers of layers, making it necessary to align the data sets using reference configurations.  Starting with a Li-terminated slab model, a smaller Ni-terminated model can be obtained by removing Li and O atoms from the surface regions (corresponding to delithiation and oxygen release). Thus, the surface energy of such a model can be aligned with that of a larger Ni-terminated slab model to achieve consistency.

This is encoded in the config file as `alignments`:

In [ ]:
print(json.dumps(config['alignments'], indent=2))

This configuration defines the "li" data set as the reference to which the "ni" data set is aligned.  For both data sets the corresponding reference entry is specified using its unique phase ID.

## 3. Load and inspect candidate phases

Let us inspect the DFT data file of the "li" data set (i.e., based on the Li-terminated slab model).

In [ ]:
table_path = configuration_path.parent / config["datasets"][0]["path"]
with open(table_path) as fp:
    print(fp.read())

The first row contains the column headers `phase_id`, `Li`, `Ni`, `O`, `dft_energy_ev`, and `surface_area_angstrom2`.  The subsequent rows contain the corresponding data, i.e., a unique phase ID (arbitrary string without spaces), the atom counts for all species, the total DFT energy for the entire slab cell, and surface area.

The second ("ni") dataset is returned as an aligned, non-mutating view because the two slab families use different energy gauges.

In [ ]:
datasets = configuration.load_datasets()
for dataset in datasets:
    print(
        dataset.dataset_id,
        type(dataset).__name__,
        f"{len(dataset.phases)} phases",
        dict(dataset.phases[0].composition),
    )

## 4. Evaluate the two-dimensional diagram

For phase $p$ with composition counts $n_{pi}$, the total grand potential is

$$\Omega_p(\mathbf{x}) = E_p^{\mathrm{DFT}} + \Delta E_p^{\mathrm{align}} - \sum_i n_{pi}\mu_i(\mathbf{x}),$$

where the data set alignment energy $\Delta E_p^{\mathrm{align}}$ is determined from the specified reference models.

The plotted intensive quantity is

$\gamma_p(\mathbf{x}) = \frac{\Omega_p(\mathbf{x})}{N_{p,\mathrm{surfaces}} A_p},$

where $A_p$ is the surface-cell area and $N_{p,\mathrm{surfaces}}$ is the number of equivalent surfaces represented by the calculated cell. 

For the construction of the phase diagram, the intensive grand-canonical surface energies are evaluated on a mesh. The diagram specification constructs the voltage/temperature mesh and retains all phases tied within the documented absolute tolerance. The parameters are configured under the `diagram` node of the config file:

In [ ]:
print(json.dumps(config['diagram'], indent=2))

The following cell evaluates the phase diagram and returns some stats. We will inspect the result in the following section.

In [ ]:
specification = configuration.diagram_specification
result = specification.evaluate(configuration.model, datasets)
print("mesh shape (temperature, voltage):", result.mesh_shape)
print(
    "energy tensor shape (phase, temperature, voltage):",
    result.surface_grand_potential_ev_per_angstrom2.shape,
)
print("candidate phases:", len(result.phase_ids))
print(
    "tie tolerance (eV/angstrom^2):",
    result.stability_tolerance_ev_per_angstrom2,
)

## 5. Inspect resolved chemical potentials and stability

The result retains the resolved chemical potential of every component. Array indices follow `(temperature index, voltage index)` because the y coordinate is stored first. Stable identities are dataset-qualified so equal local phase names cannot collide.

In [ ]:
y_index, x_index = 100, 120
dataset_result = result.dataset_results[0]
V = result.x_mesh[y_index, x_index]
T = result.y_mesh[y_index, x_index]
print(f"Potential  : {V:.1f} V")
print(f"temperature: {T:.1f} K")
print("Corresponding chemical potentials:")
for component, values in dataset_result.chemical_potentials_ev.items():
    mu = values[y_index, x_index]
    print(f"   mu_{component:2s} = {mu:-7.3f} eV")
print("stable phase(s):", result.stable_phase_ids_at(y_index, x_index))

## 6. Visualizing the phase diagram

Rendering consumes the evaluated result. Axis inversion and coloring are presentation choices; they do not change the mesh, energies, or stable-phase mask.

Defaults can be configured in the `rendering` config node:

In [ ]:
print(json.dumps(config['rendering'], indent=2))

However, the returned Matplotlib objects remain under the caller's control, and the configuration can be overwritten. Black phase boundaries are enabled by default. The `boundary_color` argument accepts any Matplotlib color, `boundary_linewidth` sets the width in points, and `boundary_color=None` disables the overlay. These choices affect only the plot; they do not modify the numerical phase-diagram result.

In [ ]:
figure, axes, colorbar = plot_phase_diagram(
    result,
    coloring=configuration.create_coloring(),
    cmap=configuration.colormap,
    invert_x_axis=configuration.invert_x_axis,
    invert_y_axis=configuration.invert_y_axis,
    boundary_color="black",
    boundary_linewidth=1.0,
)
axes.set_title("LiNiO2 (001), SCAN+rVV10+U")

## Questions worth asking about the API

As you work through this notebook, consider whether the separation among configuration, model, datasets, diagram specification, numerical result, and rendering feels natural. In particular:

- Is it easy to discover why a component is independent or dependent?
- Are total-cell, per-surface-cell, and per-area energies clearly distinguished?
- Is explicit dataset alignment understandable from the returned objects?
- Is tie-preserving stability accessible enough for scientific analysis?
- Should common inspection operations have higher-level convenience methods?

In [ ]:
plt.close(figure)